In [1]:
import os
import tkinter as tk
from tkinter import filedialog, messagebox, ttk, scrolledtext
from pdf2image import convert_from_path
import pytesseract
from docx import Document

# تفعيل مسار محرك Tesseract OCR
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

class AdvancedPDFConverter:
    def __init__(self, root):
        self.root = root
        self.root.title("محرر ومحول PDF المسحوح ضوئياً إلى Word")
        self.root.geometry("700x600")
        self.root.minsize(600, 500)

        self.pdf_path = tk.StringVar()

        self.create_widgets()

    def create_widgets(self):
        # 1. إطار اختيار الملف
        top_frame = tk.Frame(self.root, pady=10, padx=10)
        top_frame.pack(fill="x")

        lbl_file = tk.Label(top_frame, text="ملف PDF:", font=("Arial", 11))
        lbl_file.pack(side="right", padx=5)

        self.entry_path = tk.Entry(top_frame, textvariable=self.pdf_path, font=("Arial", 10), width=50)
        self.entry_path.pack(side="right", padx=5, expand=True, fill="x")

        btn_browse = tk.Button(top_frame, text="تصفح...", command=self.browse_file, font=("Arial", 10))
        btn_browse.pack(side="right", padx=5)

        # 2. زر بدء استخراج النصوص وشريط التقدم
        action_frame = tk.Frame(self.root, pady=5, padx=10)
        action_frame.pack(fill="x")

        self.btn_ocr = tk.Button(action_frame, text="1. استخراج النص (OCR)", command=self.extract_text, font=("Arial", 11, "bold"), bg="#2196F3", fg="white")
        self.btn_ocr.pack(side="right", padx=5)

        self.progress = ttk.Progressbar(action_frame, orient="horizontal", mode="determinate")
        self.progress.pack(side="right", fill="x", expand=True, padx=10)

        # 3. تسمية توضيحية لمنطقة التعديل
        lbl_edit = tk.Label(self.root, text="النص المستخرج (يمكنك التعديل والكتابة هنا مباشرة):", font=("Arial", 11, "bold"), anchor="e")
        lbl_edit.pack(fill="x", padx=10, pady=(10, 2))

        # 4. منطقة تعديل النصوص (يدعم لغة عربية محاذاة لليمين)
        self.text_editor = scrolledtext.ScrolledText(self.root, font=("Arial", 12), undo=True)
        self.text_editor.pack(fill="both", expand=True, padx=10, pady=5)

        # ضبط اتجاه النص العربي لليمين
        self.text_editor.tag_configure("rtl", justify="right")

        # 5. زر حفظ ملف الوورد
        bottom_frame = tk.Frame(self.root, pady=10)
        bottom_frame.pack(fill="x")

        self.btn_save = tk.Button(bottom_frame, text="2. حفظ كملف Word (.docx)", command=self.save_to_word, font=("Arial", 12, "bold"), bg="#4CAF50", fg="white", state="disabled")
        self.btn_save.pack(pady=5)

    def browse_file(self):
        file_selected = filedialog.askopenfilename(filetypes=[("PDF Files", "*.pdf")])
        if file_selected:
            self.pdf_path.set(file_selected)

    def extract_text(self):
        pdf_file = self.pdf_path.get()
        if not pdf_file:
            messagebox.showwarning("تنبيه", "الرجاء اختيار ملف PDF أولاً.")
            return

        try:
            self.btn_ocr.config(state="disabled")
            self.text_editor.delete("1.0", tk.END)
            self.root.update_idletasks()

            # تحويل PDF لصور (تأكد من فك ضغط مجلد poppler في المجلد المذكور أدناه أو قم بتغييره لمسارك)
            # إذا وضعت مجلد الـ poppler في القرص C مباشرة سيكون المسار الافتراضي كالتالي:
            poppler_bin_path = r'C:\poppler\Library\bin'
            
            pages = convert_from_path(pdf_file, dpi=200, poppler_path=poppler_bin_path)
            total_pages = len(pages)
            self.progress["maximum"] = total_pages

            full_text = ""
            for index, page in enumerate(pages):
                text = pytesseract.image_to_string(page, lang='ara+eng')
                full_text += text + "\n\n"
                
                self.progress["value"] = index + 1
                self.root.update_idletasks()

            self.text_editor.insert(tk.END, full_text)
            self.text_editor.tag_add("rtl", "1.0", tk.END)
            
            self.btn_save.config(state="normal")
            messagebox.showinfo("اكتمل الاستخراج", "تم استخراج النص بنجاح. يمكنك الآن مراجعته وتعديله في المربع أدناه.")

        except Exception as e:
            messagebox.showerror("خطأ", f"حدث خطأ أثناء معالجة الملف:\n{str(e)}\n\nتأكد من تثبيت Poppler ووضع ملفاته في المسار الصحيح.")
        
        finally:
            self.btn_ocr.config(state="normal")

    def save_to_word(self):
        edited_text = self.text_editor.get("1.0", tk.END).strip()
        
        if not edited_text:
            messagebox.showwarning("تنبيه", "مربع النص فارغ! لا يوجد شيء لحفظه.")
            return

        default_name = os.path.splitext(os.path.basename(self.pdf_path.get()))[0] + "_معدل.docx"
        output_file = filedialog.asksaveasfilename(defaultextension=".docx", initialfile=default_name, filetypes=[("Word Documents", "*.docx")])
        
        if output_file:
            try:
                doc = Document()
                for line in edited_text.split('\n'):
                    doc.add_paragraph(line)
                
                doc.save(output_file)
                messagebox.showinfo("نجاح", f"تم حفظ ملف الوورد بنجاح في:\n{output_file}")
            except Exception as e:
                messagebox.showerror("خطأ", f"فشل حفظ الملف:\n{str(e)}")

if __name__ == "__main__":
    root = tk.Tk()
    app = AdvancedPDFConverter(root)
    root.mainloop()